In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [4]:
df = pd.read_csv("/content/insurance_data.csv")

In [5]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,64,59.8,1.63,3.87000,False,Mumbai,retired,Medium
3,60,117.8,1.66,50.00000,True,Lucknow,business_owner,High
4,40,70.0,1.59,28.16664,True,Bangalore,government_job,Low
2,67,114.5,1.74,0.61000,True,Mumbai,retired,High
1,51,100.6,1.68,11.99000,True,Bangalore,unemployed,High


In [6]:
df_feat = df.copy()

In [7]:
# Feature 1: BMI
df_feat["bmi"] = df_feat["weight"] / (df_feat["height"] ** 2)

In [8]:
# Feature 2: Age Group
def age_group(age):
  if age < 25:
    return "young"
  elif age < 45:
    return "adult"
  elif age < 60:
    return "middle_aged"
  return "senior"

In [9]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [10]:
# Feature 3: LifeStyle Risk
def lifestyle_risk(row):
  if row["smoker"] and row["bmi"] > 30:
    return "high"
  elif row["smoker"] or row["bmi"] > 27:
    return "medium"
  else:
    return "low"

In [11]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [12]:
tier_1_cities = ["Mumbai", "Delhi", "Bengaluru", "Hyderabad", "Chennai", "Kolkata", "Pune", "Ahmedabad"]

tier_2_cities = [
    "Surat", "Jaipur", "Lucknow", "Kanpur", "Nagpur",
    "Indore", "Thane", "Bhopal", "Visakhapatnam", "Patna",
    "Vadodara", "Ghaziabad", "Ludhiana", "Agra", "Nashik",
    "Faridabad", "Meerut", "Rajkot", "Varanasi", "Srinagar",
    "Aurangabad", "Dhanbad", "Amritsar", "Navi Mumbai", "Prayagraj",
    "Ranchi", "Jabalpur", "Gwalior", "Coimbatore", "Vijayawada",
    "Jodhpur", "Madurai", "Raipur", "Kota", "Guwahati",
    "Chandigarh", "Solapur", "Hubli-Dharwad", "Tiruchirappalli", "Bareilly"
]

In [13]:
# Feature 4: City tier
def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3

In [14]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [16]:
df_feat.drop(columns=["age", "weight", "height", "smoker", "city"])[["income_lpa", "occupation", "bmi", "age_group", "lifestyle_risk", "city_tier", "insurance_premium_category"]]

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
0,3.87000,retired,22.507433,senior,low,1,Medium
1,11.99000,unemployed,35.643424,middle_aged,high,3,High
2,0.61000,retired,37.818734,senior,high,1,High
3,50.00000,business_owner,42.749310,senior,high,2,High
4,28.16664,government_job,27.688778,adult,medium,3,Low


In [18]:
# Select Features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
Y = df_feat["insurance_premium_category"]


In [19]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,22.507433,senior,low,1,3.87000,retired
1,35.643424,middle_aged,high,3,11.99000,unemployed
2,37.818734,senior,high,1,0.61000,retired
3,42.749310,senior,high,2,50.00000,business_owner
4,27.688778,adult,medium,3,28.16664,government_job


In [20]:
Y

,insurance_premium_category
0,Medium
1,High
2,High
3,High
4,Low


In [21]:
# Define Catgorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numerical_features = ["bmi", "income_lpa"]

In [22]:
# Create column transformer for ONE
preprocessor = ColumnTransformer(
    transformers = [
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

In [24]:
# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [25]:
# split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [27]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

1.0

In [32]:
print(f"Rows in X_test: {len(X_test)}")

Rows in X_test: 1


In [34]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
2,37.818734,senior,high,1,0.61,retired


In [35]:
import pickle

# save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
  pickle.dump(pipeline, f)